# Skill extraction from job postings: training run

Trains two models on **SkillSpan** and compares them:
- `bert-base-cased`: generic English BERT (baseline)
- `jjzha/jobbert-base-cased`: BERT further pretrained on job postings

Each model has **two BIO taggers** (SKILL and KNOWLEDGE) so overlapping spans aren't lost.
Code: https://github.com/MSR327/skillspan-ner

**Before running:** Settings → Accelerator = **GPU T4 x2**, Internet = **On**.

## 1. Get the code

In [ ]:
!git clone -q https://github.com/MSR327/skillspan-ner.git
%cd skillspan-ner
!pip install -q -U "transformers>=5.0" "accelerate>=1.1" datasets
!pip install -q -e ".[train]"

## 2. Check the GPU is on (should list a Tesla T4)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. (Optional) Hugging Face login, to publish the model

Add your HF **write** token under *Add-ons → Secrets* with the name `HF_TOKEN`, and set `HF_USERNAME` below.
If you skip this, training still works; the model just isn't uploaded.

In [ ]:
from pathlib import Path

import yaml

HF_USERNAME = ""  # e.g. "msr327"; leave empty to skip uploading

if HF_USERNAME:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient

    login(UserSecretsClient().get_secret("HF_TOKEN"))  # fails loudly if the secret is missing
    cfg_path = Path("configs/jobbert.yaml")
    cfg = yaml.safe_load(cfg_path.read_text())
    cfg["hub_model_id"] = f"{HF_USERNAME}/jobbert-skill-extractor"
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print("Will push to", cfg["hub_model_id"])
else:
    print("No HF_USERNAME set: training only, no upload.")

## 4. Train the baseline (~15–20 min)

In [ ]:
!python -m skillex.train --config configs/bert_baseline.yaml 2>&1 | grep -vE 'it/s|s/it'

## 5. Train JobBERT (~15–20 min)

In [ ]:
!python -m skillex.train --config configs/jobbert.yaml 2>&1 | grep -vE 'it/s|s/it'

## 6. Results (test set, exact span-match)

In [ ]:
import json
from pathlib import Path

import pandas as pd

KEEP = ("f1", "precision", "recall")
rows = []
for f in sorted(Path("metrics").glob("*.json")):
    m = json.loads(f.read_text())
    scores = {k[5:]: round(v, 4) for k, v in m.items() if k.startswith("test_") and any(x in k for x in KEEP)}
    rows.append({"model": f.stem, **scores})
df = pd.DataFrame(rows).set_index("model")
df[["precision", "recall", "f1", "f1_skill", "f1_knowledge", "recall_skill", "recall_knowledge"]]

## 7. Try it on a real job post

In [ ]:
import sys

sys.path.insert(0, "src")  # editable install isn't visible to an already-running kernel

from skillex.predict import SkillExtractor

ex = SkillExtractor("outputs/jobbert/best", device="cuda")
r = ex.extract(
    "We are looking for a Data Engineer with strong Python and SQL experience. "
    "You will design ETL pipelines on AWS, collaborate with stakeholders, and "
    "mentor junior engineers. Knowledge of Spark and Airflow is a plus."
)
print("SKILLS:   ", r["skills"])
print("KNOWLEDGE:", r["knowledge"])